In [1]:

import chromadb
import numpy as np
import pandas as pd
from renumics import spotlight


def chroma_to_dataframe(
    persist_directory: str,
    collection_name: str,
    max_points: int = 5000,
) -> pd.DataFrame:
    client = chromadb.PersistentClient(path=persist_directory)
    collection = client.get_collection(collection_name)

    count = collection.count()
    if count == 0:
        raise ValueError(f"La collection '{collection_name}' est vide.")

    limit = min(max_points, count)

    results = collection.get(
        limit=limit,
        include=["documents", "embeddings", "metadatas"],
    )

    ids = results["ids"]
    docs = results["documents"]
    embs = results["embeddings"]
    metas = results["metadatas"]

    rows = []
    for _id, doc, emb, meta in zip(ids, docs, embs, metas):
        row = {
            "id": _id,
            "document": doc,
            "embedding": np.array(emb, dtype=np.float32),
        }
        meta = meta or {}
        for k, v in meta.items():
            if k not in row:
                row[k] = v
        rows.append(row)

    df = pd.DataFrame(rows)
    return df


def visualize_with_spotlight(df: pd.DataFrame) -> None:
    dtype = {
        "embedding": spotlight.Embedding,
    }
    spotlight.show(df, dtype=dtype)


def main():
    persist_directory = "../data/chroma_db"
    collection_name = "legal_documents"

    df = chroma_to_dataframe(
        persist_directory=persist_directory,
        collection_name=collection_name,
        max_points=5000,
    )

    print(f"✅ DataFrame construit avec {len(df)} lignes.")
    visualize_with_spotlight(df)


if __name__ == "__main__":
    main()


2025-11-24 03:03:51.887 | WARNING  | renumics.spotlight.analysis.analyzers.cleanlab:<module>:20 - Cleanlab analyzer requires `cleanlab` to be installed.
2025-11-24 03:03:51.888 | WARNING  | renumics.spotlight.analysis.analyzers.cleanvision:<module>:24 - Cleanvision analyzer requires `cleanvision` to be installed.


✅ DataFrame construit avec 8 lignes.
